# positive_control_gen — generate the 302 screen rows with the memorization adapters

Minimal edit of `lora_arm_step1000`: the dataset is `screen` rather than `test`, and the adapters
are the memorization checkpoints, which were trained on these same 302 rows. Sampler, resolution,
negative prompt, seed and the resumable engine are unchanged, so output is comparable to
`gen/screen/baseline`, already generated and scored at these settings.

Two checkpoints are generated by default. Step 100 sits at 223% of the arm's step-4000 adapter
norm and 5.3 epochs; step 500 at 656% and 26.5 epochs. Scoring both separates a control that
failed to memorise from one trained past its peak: a failure at step 500 alone would not
distinguish them. Add 200, 300 or 400 to `STEPS` if the two ends disagree.

The pipeline is rebuilt per checkpoint rather than swapping adapters in place, which keeps the
construction identical to every other adapter arm. Roughly 2 hours per checkpoint at the measured
23.5 s/image. Resumable via skip-if-exists.

## 1. Repo

In [1]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

# peft's LoRA injection walks dispatch_torchao, which raises rather than returning False when torchao
# predates 0.16 (Colab ships 0.10). torchao is unused by this bf16 path; removing it lets injection
# proceed. Same fix the converter and training notebooks use. Guarded, so it is a no-op once gone.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

sys.path.insert(0, REPO)
os.chdir(REPO)

installed.
torchao removed


## 2. Drive, run config, adapter paths

In [2]:
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET = "screen"            # the 302 rows the control was trained on
RANK    = 8                   # the rank the headline arm reports
STEPS   = [100, 500]          # 5.3 and 26.5 epochs; add 200/300/400 if the two ends disagree
TRAIN   = "positive_control_r8"

EXP = f"{DRIVE}/lora/experiments"
PLAN = []
for STEP in STEPS:
    LORA_DIR = f"{EXP}/{TRAIN}/checkpoint-{STEP}/transformer_lora"
    LORA     = f"{LORA_DIR}/pytorch_lora_weights.safetensors"
    assert os.path.exists(LORA), f"adapter missing, run positive_control_convert first: {LORA}"
    RUN = f"{TRAIN}_step{STEP}"
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}"
    os.makedirs(GEN, exist_ok=True)
    # guard: GEN is derived from RUN alone, so a mismatched STEP would silently write into another
    # checkpoint's directory, where skip-if-exists would then score the wrong images
    assert f"checkpoint-{STEP}" in LORA_DIR and f"step{STEP}" in GEN, "adapter step and output directory disagree"
    n_have = len([f for f in os.listdir(GEN) if f.endswith(".png")])
    PLAN.append(dict(step=STEP, run=RUN, lora_dir=LORA_DIR, gen=GEN))
    print(f"step {STEP:>4d} -> {GEN}   already generated: {n_have}")
print(f"\ndata: {DRIVE}")

Mounted at /content/drive
step  100 -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/screen/positive_control_r8_step100   already generated: 0
step  500 -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/screen/positive_control_r8_step500   already generated: 0

data: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data


## 3. Images, split files

In [3]:
import subprocess, json
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"
def n_files(d): return sum(len(f) for _,_,f in os.walk(d)) if os.path.isdir(d) else 0
if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train","val","test","screen","screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
missing = [p for r in rows for p in r["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
assert len(rows) == 302, f"expected 302 screen rows, got {len(rows)}"
print(f"{DATASET}: {len(rows)} rows, all reference images present")

# the comparator: same rows, same settings, already scored
BASE_GEN = f"{DRIVE}/gen/{DATASET}/baseline"
print(f"comparator: {BASE_GEN} "
      f"({len([f for f in os.listdir(BASE_GEN) if f.endswith('.png')]) if os.path.isdir(BASE_GEN) else 0} images)")

restored 37063 / 37063
screen: 302 rows, all reference images present
comparator: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/screen/baseline (302 images)


## 4. Engine and run config

In [4]:
import torch, time
from accelerate import Accelerator
from PIL import Image, ImageOps
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel

MODEL = "OmniGen2/OmniGen2"
accelerator = Accelerator(mixed_precision="bf16")
dtype = torch.bfloat16

# shipped default from inference.py and omnicontext/inference.py; quality filter, not a domain negative
NEG = ("(((deformed))), blurry, over saturation, bad anatomy, disfigured, poorly drawn face, mutation, mutated, "
       "(extra_limb), (ugly), (poorly drawn hands), fused fingers, messy drawing, broken legs censor, censored, censor_bar")

def load_inputs(paths, max_side=512):   # refs downscaled; output size fixed at 512 via align_res=False
    out = []
    for p in paths:
        im = ImageOps.exif_transpose(Image.open(p).convert("RGB"))
        w, h = im.size
        s = max_side / max(w, h)
        if s < 1: w, h = int(w*s), int(h*s)
        w, h = max(16, (w//16)*16), max(16, (h//16)*16)
        out.append(im.resize((w, h), Image.LANCZOS))
    return out

def build(lora_dir):
    # same construction as inference.py load_pipeline, except that the adapter is grafted with
    # transformer.load_lora_adapter: inference.py's pipeline.load_lora_weights fails against current
    # diffusers, since OmniGen2's lora_state_dict iterates the return value of _fetch_state_dict,
    # now a (state_dict, metadata) tuple
    p = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True)
    p.transformer = OmniGen2Transformer2DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype)
    p.transformer.load_lora_adapter(lora_dir, weight_name="pytorch_lora_weights.safetensors", prefix="transformer")
    return p.to(accelerator.device)

def generate(pipe, instruction, input_images=None, negative=NEG, seed=0, steps=50,
             text_g=5.0, image_g=2.0, size=512):
    gen = torch.Generator(device=accelerator.device).manual_seed(seed)
    res = pipe(prompt=instruction, input_images=input_images,
               width=size, height=size, num_inference_steps=steps, max_sequence_length=1024,
               text_guidance_scale=text_g, image_guidance_scale=image_g, cfg_range=(0.0, 1.0),
               negative_prompt=negative, num_images_per_prompt=1, generator=gen, output_type="pil",
               align_res=False)
    return res.images[0]

# no prompt lever in this arm: the adapter is the treatment, instruction and negative are the baseline
cfg = dict(instruction=lambda r: r["instruction"], negative=NEG)
_probe = {"instruction": "<probe>", "input_images": []}
print(f"  instruction : {'unchanged' if cfg['instruction'](_probe) == '<probe>' else 'transformed'}")
print(f"  negative    : {'shipped default' if cfg['negative'] == NEG else 'custom'}")
print(f"  sampler     : 50 steps, text_g=5.0, image_g=2.0, cfg_range=(0,1), seed 0, 512x512  (fixed across all conditions)")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")


  instruction : unchanged
  negative    : shipped default
  sampler     : 50 steps, text_g=5.0, image_g=2.0, cfg_range=(0,1), seed 0, 512x512  (fixed across all conditions)


/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


## 5. Full pass (resumable)

In [5]:
import json, subprocess

for spec in PLAN:
    STEP, RUN, GEN, LORA_DIR = spec["step"], spec["run"], spec["gen"], spec["lora_dir"]
    todo = [i for i in range(len(rows)) if not os.path.exists(f"{GEN}/{i:05d}.png")]
    print(f"\n===== {RUN}: {len(rows)-len(todo)} done, {len(todo)} to generate =====", flush=True)
    if not todo:
        spec["done"] = 0
        continue

    pipe = build(LORA_DIR)
    print(f"pipeline ready on {accelerator.device} with the step-{STEP} adapter", flush=True)

    # smoke: first row, checked for geometry before committing two hours
    r0 = rows[0]
    img = generate(pipe, cfg["instruction"](r0),
                   input_images=load_inputs([os.path.join(IMGS, p) for p in r0["input_images"]]),
                   negative=cfg["negative"], seed=0)
    assert img.size == (512, 512), f"expected 512x512, got {img.size}"
    print("smoke OK", img.size, flush=True)

    done, t0 = 0, time.time()
    for i in todo:
        r = rows[i]
        refs = load_inputs([os.path.join(IMGS, p) for p in r["input_images"]])
        img = generate(pipe, cfg["instruction"](r), input_images=refs, negative=cfg["negative"], seed=0)
        assert img.size == (512, 512), (i, img.size)
        img.save(f"{GEN}/{i:05d}.png")
        done += 1
        if done % 25 == 0:
            print(f"  {done}/{len(todo)} (row {i+1}/{len(rows)}), {(time.time()-t0)/done:.1f}s/img", flush=True)

    total = len([f for f in os.listdir(GEN) if f.endswith(".png")])
    print(f"{RUN}: {total}/{len(rows)} images in {GEN}")
    assert total == len(rows), "incomplete"
    spec["done"] = done
    del pipe
    torch.cuda.empty_cache()


===== positive_control_r8_step100: 0 done, 302 to generate =====


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

added_tokens.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

mllm/model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  300/302 (row 300/302), 25.9s/img


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

positive_control_r8_step500: 302/302 images in /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/screen/positive_control_r8_step500


## 6. Manifests

In [6]:
# manifest: provenance for the images in each GEN. Written only when this session generated some,
# so a skip-through or resumed run cannot stamp the wrong GPU onto images made earlier.
for spec in PLAN:
    GEN, RUN, STEP = spec["gen"], spec["run"], spec["step"]
    if not spec.get("done"):
        print(f"no images generated this session for {RUN}; manifest left as-is")
        continue
    man = {
        "dataset": DATASET, "run": RUN, "step": STEP,
        "n_images": len([f for f in os.listdir(GEN) if f.endswith(".png")]),
        "gpu": torch.cuda.get_device_name(0),
        "driver": subprocess.run("nvidia-smi --query-gpu=driver_version --format=csv,noheader",
                                 shell=True, capture_output=True, text=True).stdout.strip(),
        "torch": torch.__version__, "cuda": torch.version.cuda, "model": MODEL,
        "sampler": dict(steps=50, text_guidance_scale=5.0, image_guidance_scale=2.0,
                        cfg_range=[0.0, 1.0], max_sequence_length=1024, seed=0,
                        size=512, align_res=False),
        "negative_prompt": cfg["negative"],
        "instruction_row0": cfg["instruction"](rows[0]),
        "trained_on": "the same 302 rows; this is a memorization positive control",
        "bit_reproducible": False,
        "written": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(f"{GEN}/manifest.json", "w") as f:
        json.dump(man, f, indent=2)
    print("manifest written:", f"{GEN}/manifest.json")
print("\nnext: score_runs_positive_control")

manifest written: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/screen/positive_control_r8_step100/manifest.json
manifest written: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/screen/positive_control_r8_step500/manifest.json

next: score_runs_positive_control
